# K1 · Weather generation and visual QA

**CPU is enough.** Enable Internet, optionally attach Cityscapes train images, disparity and camera calibration, then Run All. The notebook first previews a procedural scene with known synthetic distances, then selects up to3source TRAIN images with a fixed seed. No ACDC is loaded and no training is started.

Fog requires metric distance. Missing distance skips fog on real images; no fake depth is invented. Stereo invalids use the explicitly selected far-fill approximation, recorded per image. Rain/snow are 2D overlays and night is a linearized-sRGB sensor approximation. Severity1–5 is an engineering preset, not a calibrated real-weather scale.

In [ ]:
"""Standard-library source bootstrap shared by the two DG model notebooks."""
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile


MARKERS=('scripts/run_dg_suite.py','src/training/checkpoints.py','src/models/registry.py')


def valid_source(path):
    path=Path(path)
    if path.is_dir():return all((path/n).is_file() for n in MARKERS)
    if path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:return set(MARKERS).issubset(z.namelist())
    return False


def resolve_source(hint,input_root):
    hint=Path(hint);input_root=Path(input_root)
    if valid_source(hint):return hint
    search_root=hint if hint.is_dir() and hint.resolve()!=input_root.resolve() else input_root
    anchor=PurePosixPath(MARKERS[0])
    folders=sorted({p.parents[len(anchor.parts)-1] for p in search_root.rglob(anchor.name) if valid_source(p.parents[len(anchor.parts)-1])})
    candidates=folders or [p for p in sorted(search_root.rglob('*.zip')) if valid_source(p)]
    if len(candidates)!=1:raise ValueError(f'Need exactly one current code source; found{len(candidates)}: {candidates}. Set CODE_INPUT explicitly.')
    return candidates[0]


def extract_safe(archive,destination):
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        seen=set()
        for item in z.infolist():
            name=PurePosixPath(item.filename.replace('\\','/'))
            if name.is_absolute() or '..' in name.parts or ':' in str(name) or stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:
                raise ValueError('Unsafe archive member')
            key=str(name).rstrip('/').casefold()
            if key in seen:raise ValueError('Duplicate archive member')
            seen.add(key)
            if not (destination/str(name)).resolve().is_relative_to(destination):raise ValueError('Unsafe archive destination')
        z.extractall(destination)


def copy_source(source,destination):
    source=Path(source);destination=Path(destination)
    if not valid_source(source):raise ValueError('Stale or missing suite code')
    if source.is_file():extract_safe(source,destination)
    else:
        destination.mkdir(parents=True,exist_ok=True)
        for name in ('src','scripts','configs','tests','third_party'):
            if (source/name).exists():shutil.copytree(source/name,destination/name,dirs_exist_ok=True)
        for name in ('requirements.txt','pyproject.toml','THIRD_PARTY_NOTICES.md'):
            if (source/name).is_file():shutil.copy2(source/name,destination/name)

import os,sys,subprocess
CODE_INPUT=None  # Optional uploaded project-code.zip/folder; None clones the K1 branch
REPO=Path('/kaggle/working/k1_code')
OUT=Path('/kaggle/working/k1_preview')
assert not REPO.exists() and not OUT.exists(), 'Use a fresh session or new output paths'
if CODE_INPUT is None:
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch','feat/k1-weather-augmentation',
                    'https://github.com/MinhToan1001/adverse-weather-segmentation.git',str(REPO)],check=True)
else:
    copy_source(resolve_source(Path(CODE_INPUT),Path('/kaggle/input')),REPO)
assert (REPO/'scripts/preview_weather.py').is_file(),'Need the K1 code branch/bundle'
os.chdir(REPO)
sys.path.insert(0,str(REPO))
def run(*args):
    subprocess.run([sys.executable,'-u',*map(str,args)],cwd=REPO,check=True)
run('-m','pip','install','numpy>=1.26,<3','Pillow>=10,<12')


In [ ]:
from IPython.display import display
from PIL import Image
run('-m','scripts.preview_weather','--output',OUT/'analytic_demo')
display(Image.open(OUT/'analytic_demo/preview.png'))
print('This first scene has analytically specified synthetic distances; it is software QA only.')


In [ ]:
import numpy as np
INPUT=Path('/kaggle/input')
NUM_IMAGES=3
SEED=42
IMAGE_PATH=None  # Optional single RGB image; otherwise randomly select Cityscapes TRAIN frames
DISTANCE_NPY=None  # Optional single-image camera-ray metric distance; not arbitrary relative depth
DISPARITY_PATH=None  # Optional single-image original Cityscapes disparity PNG
CAMERA_PATH=None  # Matching camera JSON
INVALID_POLICY='far'  # EXPLICIT approximation: invalid stereo -> 200m; fraction recorded in metadata
if IMAGE_PATH is not None:
    images=[Path(IMAGE_PATH)]
else:
    available=sorted(p for p in INPUT.rglob('*_leftImg8bit.png')
                     if p.parent.parent.name=='train' and p.parent.parent.parent.name=='leftImg8bit')
    chosen=np.random.default_rng(SEED).choice(len(available),size=min(NUM_IMAGES,len(available)),replace=False)
    images=[available[int(i)] for i in chosen]
print('Source train images:',len(images))
if not images:
    print('Attach Cityscapes leftImg8bit/train to preview real source images. Analytic demo above is complete.')


In [ ]:
import json
reports=[]
for index,image in enumerate(images):
    stem=image.stem.removesuffix('_leftImg8bit')
    disparity=Path(DISPARITY_PATH) if DISPARITY_PATH is not None else None
    camera=Path(CAMERA_PATH) if CAMERA_PATH is not None else None
    if DISTANCE_NPY is None and disparity is None and camera is None:
        ds=sorted(INPUT.rglob(stem+'_disparity.png'))
        cs=sorted(INPUT.rglob(stem+'_camera.json'))
        assert len(ds)<=1 and len(cs)<=1,'Ambiguous depth inputs; set explicit single-image paths'
        if ds and cs:
            disparity,camera=ds[0],cs[0]
    destination=OUT/f'source_{index:02d}'
    args=['-m','scripts.preview_weather','--image',image,'--output',destination,'--seed',SEED]
    if DISTANCE_NPY is not None:
        assert len(images)==1,'DISTANCE_NPY belongs to one image'
        args+=['--distance',DISTANCE_NPY]
    elif disparity is not None or camera is not None:
        assert disparity is not None and camera is not None,'Need both disparity and camera'
        args+=['--disparity',disparity,'--camera',camera,'--invalid-policy',INVALID_POLICY]
    else:
        args+=['--skip-fog']
        print('No metric distance:',image.name,'-> fog explicitly skipped; night/rain/snow still previewed')
    run(*args)
    display(Image.open(destination/'preview.png'))
    report=json.loads((destination/'metadata.json').read_text())
    reports.append(report)
    print('Distance provenance:',report['distance'])


In [ ]:
from IPython.display import FileLink
archive=Path(shutil.make_archive(str(OUT),'zip',root_dir=OUT))
display(FileLink(str(archive)))
print('Download previews + metadata for inspection. No model has been trained.')


Before training: inspect randomly selected examples at all severities; verify depth/units/invalid fractions; compare against ordinary augmentation or Foggy Cityscapes under a controlled experiment. Do not use this same generator as the sole test benchmark. See docs/K1_WEATHER.md for equations, limits and integration.